# Model Evaluation & Error Analysis — Module 1 Foundations

Deep dive into model evaluation techniques:
- Cross-validation strategies
- Metric selection for business problems
- Error analysis and debugging
- Learning curves and validation curves
- Model calibration
- Bias-variance tradeoff visualization
- Subgroup/fairness analysis

**Learning Outcomes:**
1. Choose appropriate evaluation metrics
2. Diagnose model failures systematically
3. Visualize model behavior and limitations
4. Communicate model performance to stakeholders

In [ ]:
# Imports and Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    learning_curve, validation_curve, cross_validate
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve, brier_score_loss
)
from sklearn.inspection import permutation_importance

np.random.seed(42)
sns.set_style("whitegrid")

In [ ]:
# Load Data and Build Baseline Pipeline
adult = fetch_openml(data_id=1590, as_frame=True, parser='pandas')
df = adult.frame.copy()

target_col = 'class'
feature_cols = [c for c in df.columns if c != target_col]

numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_features)
    ]
)

# Baseline model
base_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
base_model.fit(X_train, y_train)
y_pred = base_model.predict(X_test)
y_proba = base_model.predict_proba(X_test)[:, 1]

print("Baseline model trained.")

In [ ]:
# Cross-Validation with Multiple Metrics
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'roc_auc']
cv_results = cross_validate(base_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=True)

print("=== Cross-Validation Results ===")
for metric in scoring:
    train_scores = cv_results[f'train_{metric}']
    test_scores = cv_results[f'test_{metric}']
    print(f"{metric:20s} | Train: {train_scores.mean():.4f}±{train_scores.std():.4f} | Test: {test_scores.mean():.4f}±{test_scores.std():.4f}")

# Detailed test set evaluation
print("\n=== Test Set Metrics ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1:        {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print(f"Brier:     {brier_score_loss(y_test == '>50K', y_proba):.4f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix and Error Analysis
cm = confusion_matrix(y_test, y_pred)
classes = base_model.classes_

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Absolute counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# Error analysis: which samples are misclassified?
test_df = X_test.copy()
test_df['actual'] = y_test.values
test_df['predicted'] = y_pred
test_df['prob_>50K'] = y_proba
test_df['correct'] = test_df['actual'] == test_df['predicted']

errors = test_df[~test_df['correct']]
print(f"Total errors: {len(errors)} out of {len(test_df)} ({len(errors)/len(test_df):.2%})")

# Error types
print("\nError breakdown:")
print(errors.groupby(['actual', 'predicted']).size())

# High confidence errors
high_conf_errors = errors[errors['prob_>50K'] > 0.9]
low_conf_errors = errors[errors['prob_>50K'] < 0.1]
print(f"\nHigh confidence errors (>0.9): {len(high_conf_errors)}")
print(f"Low confidence errors (<0.1): {len(low_conf_errors)}")

In [ ]:
# Learning Curves
print("Computing learning curves...")
train_sizes, train_scores, val_scores = learning_curve(
    base_model, X_train, y_train, cv=cv, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', label='Training score', color='blue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
plt.plot(train_sizes, val_mean, 'o-', label='Validation score', color='red')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.title('Learning Curves (Random Forest)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Validation Curves - n_estimators
print("Computing validation curves...")
param_range = [10, 25, 50, 100, 200, 300]
train_scores, val_scores = validation_curve(
    base_model, X_train, y_train,
    param_name='classifier__n_estimators',
    param_range=param_range,
    cv=cv, scoring='accuracy', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(param_range, train_mean, 'o-', label='Training score', color='blue')
plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
plt.plot(param_range, val_mean, 'o-', label='Validation score', color='red')
plt.fill_between(param_range, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
plt.xlabel('n_estimators')
plt.ylabel('Accuracy')
plt.title('Validation Curve: n_estimators')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ROC and Precision-Recall Curves
y_binary = (y_test == '>50K').astype(int)

# ROC Curve
fpr, tpr, roc_thresholds = roc_curve(y_binary, y_proba)
roc_auc = roc_auc_score(y_binary, y_proba)

# Precision-Recall Curve
precision, recall, pr_thresholds = precision_recall_curve(y_binary, y_proba)
pr_auc = auc(recall, precision) if 'auc' in dir() else np.trapz(precision, recall)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
axes[0].plot(fpr, tpr, label=f'RF (AUC = {roc_auc:.3f})', color='darkorange', lw=2)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR Curve
axes[1].plot(recall, precision, label=f'RF (AUC = {pr_auc:.3f})', color='darkorange', lw=2)
# Baseline (random classifier)
baseline = y_binary.mean()
axes[1].axhline(y=baseline, color='k', linestyle='--', label=f'Random (AUC = {baseline:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Threshold analysis
print("=== Threshold Analysis ===")
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_thresh = (y_proba >= thresh).astype(int)
    tp = ((y_pred_thresh == 1) & (y_binary == 1)).sum()
    fp = ((y_pred_thresh == 1) & (y_binary == 0)).sum()
    fn = ((y_pred_thresh == 0) & (y_binary == 1)).sum()
    tn = ((y_pred_thresh == 0) & (y_binary == 0)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"Threshold {thresh}: Precision={prec:.3f}, Recall={rec:.3f}, TP={tp}, FP={fp}, FN={fn}")

In [ ]:
# Model Calibration
from sklearn.calibration import calibration_curve

# Calibration curve for base model
prob_true, prob_pred = calibration_curve(y_binary, y_proba, n_bins=10, strategy='uniform')

plt.figure(figsize=(8, 5))
plt.plot(prob_pred, prob_true, 'o-', label='Random Forest', color='darkorange')
plt.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curve (Reliability Diagram)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Calibrated model
print("Training calibrated model...")
calibrated_model = CalibratedClassifierCV(base_model, method='isotonic', cv=3)
calibrated_model.fit(X_train, y_train)

y_proba_cal = calibrated_model.predict_proba(X_test)[:, 1]
prob_true_cal, prob_pred_cal = calibration_curve(y_binary, y_proba_cal, n_bins=10, strategy='uniform')

# Compare Brier scores
brier_orig = brier_score_loss(y_binary, y_proba)
brier_cal = brier_score_loss(y_binary, y_proba_cal)

print(f"Original Brier Score: {brier_orig:.4f}")
print(f"Calibrated Brier Score: {brier_cal:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(prob_pred, prob_true, 'o-', label=f'Random Forest (Brier={brier_orig:.3f})', color='darkorange')
plt.plot(prob_pred_cal, prob_true_cal, 's-', label=f'Calibrated (Brier={brier_cal:.3f})', color='green')
plt.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Permutation Feature Importance
print("Computing permutation importance (this may take a moment)...")
perm_importance = permutation_importance(
    base_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1, scoring='accuracy'
)

# Get feature names
preprocessor_fitted = base_model.named_steps['preprocessor']
cat_encoder = preprocessor_fitted.named_transformers_['cat'].named_steps['encoder']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)

perm_df = pd.DataFrame({
    'feature': all_feature_names,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

print("\n=== Top 15 Permutation Importances ===")
print(perm_df.head(15))

plt.figure(figsize=(8, 6))
top15 = perm_df.head(15).iloc[::-1]
plt.barh(range(len(top15)), top15['importance_mean'], xerr=top15['importance_std'], capsize=3)
plt.yticks(range(len(top15)), top15['feature'])
plt.xlabel('Permutation Importance (Accuracy Drop)')
plt.title('Top 15 Features by Permutation Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Subgroup Analysis (Fairness Check)
# Analyze performance across sensitive attributes
sensitive_attrs = ['sex', 'race']

print("=== Subgroup Performance Analysis ===\n")

for attr in sensitive_attrs:
    if attr in X_test.columns:
        print(f"--- {attr} ---")
        subgroups = X_test[attr].unique()
        
        for subgroup in subgroups:
            mask = X_test[attr] == subgroup
            if mask.sum() > 10:  # Only report if enough samples
                sub_y_true = y_test[mask]
                sub_y_pred = y_pred[mask]
                sub_y_proba = y_proba[mask]
                
                acc = accuracy_score(sub_y_true, sub_y_pred)
                prec = precision_score(sub_y_true, sub_y_pred, average='weighted', zero_division=0)
                rec = recall_score(sub_y_true, sub_y_pred, average='weighted', zero_division=0)
                f1 = f1_score(sub_y_true, sub_y_pred, average='weighted', zero_division=0)
                auc_score = roc_auc_score((sub_y_true == '>50K').astype(int), sub_y_proba) if len(sub_y_true.unique()) > 1 else np.nan
                
                print(f"  {subgroup:20s} | n={mask.sum():4d} | Acc={acc:.3f} | P={prec:.3f} | R={rec:.3f} | F1={f1:.3f} | AUC={auc_score:.3f}")
        print()

# Demographic Parity Difference (simplified)
print("=== Demographic Parity (Positive Prediction Rate) ===")
for attr in sensitive_attrs:
    if attr in X_test.columns:
        print(f"\n{attr}:")
        for subgroup in X_test[attr].unique():
            mask = X_test[attr] == subgroup
            pos_rate = (y_pred[mask] == '>50K').mean()
            print(f"  {subgroup:20s}: {pos_rate:.3f}")

## Summary

**What we covered:**
✓ Multi-metric cross-validation (accuracy, precision, recall, F1, ROC-AUC)
✓ Confusion matrix analysis and error categorization
✓ Learning curves (diagnose bias/variance, data sufficiency)
✓ Validation curves (hyperparameter sensitivity)
✓ ROC and Precision-Recall curves with threshold analysis
✓ Model calibration (reliability diagrams, Brier score, isotonic calibration)
✓ Permutation feature importance (model-agnostic)
✓ Subgroup/fairness analysis across sensitive attributes

**Key Takeaways:**
- **Learning curves**: Gap between train/val = overfitting; both low = underfitting
- **Validation curves**: Find sweet spot before overfitting
- **PR curves**: Better than ROC for imbalanced data
- **Calibration**: Critical when probabilities drive decisions
- **Permutation importance**: Works for any model, captures interactions
- **Subgroup analysis**: Essential for responsible AI

**Next Steps:**
- SHAP values for local explanations
- Counterfactual fairness testing
- Model cards documentation
- A/B testing framework design
- Continuous evaluation in production